# Salary Prediction — Live Analytics Stream Test (VM)

Starts the real-time analytics stream (`src/streaming/analytics_stream.py`, PLAN.md §15), which reads `developer_events` and publishes windowed aggregates (`event_counts`, `salary_breakdown`, `technology_counts`) to `salary_analytics`.

**This notebook needs a Spark session with the Kafka connector**, same requirement as `run_prediction_stream.ipynb` — launch Jupyter via `scripts/start_kafka_jupyter.sh` first, not a plain `pyspark`/`jupyter notebook`.

**Prerequisites:** Kafka broker running, all 5 topics created. **Run cells top to bottom**, then see Section 5 for how to actually feed it events.

## 1. Path and working directory

In [ ]:
import sys, os

PROJECT_ROOT = "/home/linuxu/project"  # adjust if this VM's checkout lives elsewhere

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

## 1a. Install missing Python packages (one-time)

If a later cell fails with `ModuleNotFoundError`, add `%pip install <package-name>` here and re-run from the top after restarting the kernel.

In [ ]:
%pip install python-dotenv

## 2. `.env` check

In [ ]:
if not os.path.exists(".env"):
    import subprocess
    subprocess.run(["cp", ".env.example", ".env"])
    print("Created .env from .env.example.")

print(open(".env").read())

Confirm `KAFKA_BOOTSTRAP_SERVERS`, `KAFKA_DATASET_TOPIC`, `KAFKA_ANALYTICS_TOPIC`, `KAFKA_DEAD_LETTER_TOPIC` above match what you actually created on the VM.

## 3. Spark session — reuse the existing one (Kafka already works there)

Same reasoning as `run_prediction_stream.ipynb`: don't stop/recreate the session or force `spark.jars.packages` here — reuse whatever session this Kafka-enabled Jupyter server already has.

In [ ]:
from src.common.spark_session import get_spark_session
from config import settings

spark = get_spark_session(app_name="SalaryAnalyticsStream", with_kafka=False)
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("spark.master:", spark.sparkContext.master)
print("KAFKA_BOOTSTRAP_SERVERS  =", settings.KAFKA_BOOTSTRAP_SERVERS)
print("KAFKA_DATASET_TOPIC      =", settings.KAFKA_DATASET_TOPIC)
print("KAFKA_ANALYTICS_TOPIC    =", settings.KAFKA_ANALYTICS_TOPIC)
print("KAFKA_DEAD_LETTER_TOPIC  =", settings.KAFKA_DEAD_LETTER_TOPIC)

## 3a. Preflight check: is the Kafka connector actually available?

This can only be fixed by *how Jupyter itself was launched* — no code in this notebook can add the connector to an already-running session. If this cell fails, **stop here**: close this tab, run `scripts/start_kafka_jupyter.sh` in a terminal, and reopen this notebook from the NEW server it starts.

In [ ]:
try:
    (
        spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
        .option("subscribe", settings.KAFKA_DATASET_TOPIC)
        .load()
    )
    print("Kafka connector is available in this session - safe to proceed.")
except Exception as exc:
    if "Failed to find data source: kafka" in str(exc):
        print("=" * 70)
        print("KAFKA CONNECTOR NOT AVAILABLE IN THIS SESSION")
        print("=" * 70)
        print()
        print("This notebook's Spark session was started without the Kafka")
        print("connector package. This can ONLY be fixed by relaunching Jupyter")
        print("itself with the connector on the command line - no notebook code")
        print("can add it to an already-running session (PLAN.md section 23, #22).")
        print()
        print("Fix: close this tab, then in a terminal run:")
        print()
        print("    scripts/start_kafka_jupyter.sh")
        print()
        print("Then open THIS notebook again from the NEW server it prints,")
        print("and re-run from the top cell.")
        print("=" * 70)
        raise RuntimeError("Kafka connector not available - see instructions printed above.") from None
    raise

## 4. Start the analytics stream (non-blocking)

Starts all four streaming queries (dead letters + the three aggregates) and returns immediately. You're responsible for stopping them yourself in the cleanup cell at the bottom.

In [ ]:
from src.streaming.analytics_stream import build_streams

dead_letter_query, event_counts_query, salary_breakdown_query, technology_counts_query = build_streams(spark)

for name, query in [
    ("dead_letter_query", dead_letter_query),
    ("event_counts_query", event_counts_query),
    ("salary_breakdown_query", salary_breakdown_query),
    ("technology_counts_query", technology_counts_query),
]:
    print(f"{name}.isActive =", query.isActive)

## 5. Feed it: run the dataset producer concurrently

This stream has nothing to aggregate until events actually arrive on `developer_events`. Open `notebooks/run_dataset_producer.ipynb` **from this same Jupyter server** (so it shares the same Kafka-enabled launch) in a separate tab, and run its Section 3 publish cell with a batch large enough to span real time — e.g. `LIMIT = 90`, `DELAY_SECONDS = 1` (~90 seconds of event time).

Windows here are 30 seconds wide with a 15-second watermark, so the first aggregated results should start appearing roughly 45+ seconds after the producer starts publishing. Start the producer, then come back here and run the next cell (re-run it if it's still empty).

## 6. Wait, then check the results

In [ ]:
import json
import time

from pyspark.sql import functions as F

time.sleep(75)

results = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", settings.KAFKA_ANALYTICS_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string").alias("value"))
)

records = [json.loads(row["value"]) for row in results.collect()]
print(f"Total analytics records so far: {len(records)}")

by_metric = {}
for record in records:
    by_metric.setdefault(record["metric"], []).append(record)

for metric, rows in by_metric.items():
    print(f"\n--- {metric} ({len(rows)} record(s)) ---")
    print(json.dumps(rows[-1], indent=2))

if not records:
    print("No analytics records yet. Make sure the dataset producer notebook has been")
    print("running for at least ~60-90 seconds of event_time so a window can close,")
    print("then re-run this cell.")

## 7. (Optional) Check the dead-letter topic

In [ ]:
dead_letters = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", settings.KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", settings.KAFKA_DEAD_LETTER_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
    .select(F.col("value").cast("string"))
)
dead_letters.show(truncate=False)

## 8. Cleanup — stop the streaming queries

In [ ]:
for query in [dead_letter_query, event_counts_query, salary_breakdown_query, technology_counts_query]:
    query.stop()
print("Stopped all streaming queries.")